In [ ]:
import torch
import os
import torch.nn.functional as F


folder_kv_base = 'sims_kv_test'
filename = 'batch_0_k_previous.pt'
path_kv_file = os.path.join(folder_kv_base, filename)

len_prompt = 128
num_block = 4
len_target = 256
size_block = int(len_target / num_block)



In [2]:
def token_nonsimilarity_score(
    sim: torch.Tensor,
    p: float = 3.0,
) -> torch.Tensor:
    """
    sim: shape [steps, layers, tokens], values in [0, 1]
    p: power mean exponent, larger => more sensitive to low-similarity cases

    Returns:
        score: shape [tokens], larger means more non-similar
    """
    assert sim.ndim == 3, f"Expected 3D tensor [steps, layers, tokens], got shape {tuple(sim.shape)}"

    # Convert similarity to non-similarity / change magnitude
    diff = 1.0 - sim

    # Power mean over step and layer
    # First raise to power p, then average over dims 0 and 1, then take 1/p power
    score = diff.pow(p).mean(dim=(0, 1)).pow(1.0 / p)

    return score


In [3]:
matrix_sim_step_layer_token = torch.load(path_kv_file)
matrix_sim_step_layer_token = F.pad(matrix_sim_step_layer_token, (0,0,0,0,1,0), value=1.0)

In [6]:
list_idx_sim_sorted = []

for id_block in range(num_block):
    position_start = len_prompt + size_block * id_block
    matrix_sim_step_layer_token_cached = matrix_sim_step_layer_token[:size_block, :, :position_start]  #(steps_block, len_cached)
    score_diff_token = token_nonsimilarity_score(matrix_sim_step_layer_token_cached)
    idx_diff_token_decending = torch.argsort(score_diff_token, descending=True)
    list_idx_sim_sorted.append({'filename': filename, 'block': id_block, 'idx': idx_diff_token_decending.tolist(), 'value': score_diff_token[idx_diff_token_decending].tolist()})
# end

In [ ]:
list_idx_sim_sorted[]

IndexError: list index out of range